# Lecture 07: 训练神经网络 (Training Neural Networks)本笔记涵盖神经网络训练的核心技术：权重初始化、批归一化、动量法、Adam优化器、梯度检验、学习率调度与合理性检查。**学习目标：**- 理解不同权重初始化策略对训练的影响- 掌握动量法与Adam优化器的数学原理- 实现梯度检验验证反向传播正确性- 设计学习率调度策略- 执行训练合理性检查**四步教学路径：** 直觉理解 -> 手动计算 -> 代码实现 -> 实验观察

## 目录1. 权重初始化：直觉与动机2. 权重初始化：手动计算3. 权重初始化：代码实现4. 权重初始化：实验观察5. 批归一化：原理与实现6. 动量法：从物理到优化7. Adam优化器：自适应学习率8. 学习率调度策略9. 梯度检验10. 训练合理性检查11. 作业与参考文献

## 1. 权重初始化：直觉与动机### 为什么不能全部初始化为零？如果所有权重初始化为相同的值（包括零），网络中的所有神经元将计算相同的输出，接收相同的梯度，进行相同的更新。这被称为**对称性问题**。### 好的初始化需要什么？1. **打破对称性**：不同神经元应有不同的初始权重2. **控制方差**：保持各层激活值的方差稳定，避免梯度爆炸或消失3. **适配激活函数**：ReLU需要不同于Sigmoid的初始化策略### 核心直觉想象信号通过多层网络传播：- 如果权重太大 -> 激活值爆炸 -> 梯度爆炸- 如果权重太小 -> 激活值消失 -> 梯度消失- **目标：保持信号方差在前向和后向传播中大致不变**

### 方差传播的数学直觉考虑一个全连接层 y = Wx + b，其中 x 有 n 个输入。假设各元素独立同分布，均值为0：Var(y_i) = sum_{j=1}^{n} Var(W_ij * x_j) = n * Var(W_ij) * Var(x_j)为了让 Var(y_i) = Var(x_j)，需要：Var(W_ij) = 1/n这就是**Xavier初始化**的核心思想！

## 2. 权重初始化：手动计算### Xavier (Glorot) 初始化适用于tanh/sigmoid激活函数：- 均匀分布范围：U(-sqrt(6/(n_in + n_out)), sqrt(6/(n_in + n_out)))- 正态分布标准差：sqrt(2/(n_in + n_out))**手动计算示例：** n_in = 512, n_out = 256- 均匀分布范围：sqrt(6/768) = 0.0884- 正态分布标准差：sqrt(2/768) = 0.0510### He (Kaiming) 初始化适用于ReLU。ReLU将一半输入置零，因此需要2倍方差补偿：- 标准差：sqrt(2/n_in)**手动计算示例：** n_in = 512- 标准差：sqrt(2/512) = 0.0625- 对比Xavier：sqrt(2/768) = 0.0510（He更大，补偿ReLU的零化效应）### 手动验证：前向传播方差变化| 初始化方法 | 第1层方差 | 第2层方差 | 第3层方差 | 趋势 ||---|---|---|---|---|| 全零 | 0 | 0 | 0 | 完全失效 || N(0,1) | 512 | 262144 | ~1.3e8 | 爆炸 || Xavier+tanh | ~1.0 | ~1.0 | ~1.0 | 稳定 || He+ReLU | ~1.0 | ~1.0 | ~1.0 | 稳定 || N(0,0.001) | 0.0005 | 2.6e-7 | ~1.3e-10 | 消失 |

## 3. 权重初始化：代码实现下面我们从零实现四种初始化方法，并用前向传播验证方差传播特性。

In [7]:
# -*- coding: utf-8 -*-import numpy as npimport matplotlibmatplotlib.use('Agg')import matplotlib.pyplot as pltnp.random.seed(42)# ============================================================# Weight Initialization Methods# ============================================================def init_zeros(n_in, n_out):    # All zeros - demonstrates symmetry breaking problem    return np.zeros((n_in, n_out))def init_normal(n_in, n_out, std=0.01):    # Small random normal - common but often poor for deep nets    return np.random.randn(n_in, n_out) * stddef init_xavier_uniform(n_in, n_out):    # Xavier/Glorot uniform - for tanh/sigmoid    limit = np.sqrt(6.0 / (n_in + n_out))    return np.random.uniform(-limit, limit, size=(n_in, n_out))def init_xavier_normal(n_in, n_out):    # Xavier/Glorot normal - for tanh/sigmoid    std = np.sqrt(2.0 / (n_in + n_out))    return np.random.randn(n_in, n_out) * stddef init_he_normal(n_in, n_out):    # He/Kaiming normal - for ReLU    std = np.sqrt(2.0 / n_in)    return np.random.randn(n_in, n_out) * stddef init_he_uniform(n_in, n_out):    # He/Kaiming uniform - for ReLU    limit = np.sqrt(6.0 / n_in)    return np.random.uniform(-limit, limit, size=(n_in, n_out))print("Weight initialization methods defined.")print("Methods: zeros, normal(0.01), xavier_uniform, xavier_normal, he_normal, he_uniform")

In [8]:
# ============================================================# Activation Functions# ============================================================def relu(x):    return np.maximum(0, x)def tanh(x):    return np.tanh(x)def sigmoid(x):    return 1.0 / (1.0 + np.exp(-np.clip(x, -500, 500)))def linear(x):    return xprint("Activation functions defined: relu, tanh, sigmoid, linear")

In [9]:
# ============================================================# Forward Propagation Variance Tracking# ============================================================def forward_track_variance(Ws, x, activation):    # Track activation variance through layers    stats = []    a = x.copy()    stats.append((0, np.mean(a), np.var(a)))        for i, W in enumerate(Ws):        z = a @ W        a = activation(z)        stats.append((i + 1, np.mean(a), np.var(a)))        return stats# Build a 10-layer network, each layer 512 unitsn_layers = 10n_units = 512x = np.random.randn(1000, n_units)  # batch of 1000# Test with different initializationsconfigs = {    'Zeros': (init_zeros, tanh),    'Normal(0.01)': (init_normal, tanh),    'Xavier': (init_xavier_normal, tanh),    'He': (init_he_normal, relu),    'Standard N(0,1)': (lambda a, b: np.random.randn(a, b), relu),}print("Forward propagation tracker defined.")print(f"Network: {n_layers} layers, {n_units} units each")print(f"Input batch: {x.shape}")

## 4. 权重初始化：实验观察让我们可视化不同初始化方法在深层网络中的方差传播行为。

In [11]:
# ============================================================# Experiment: Compare initialization methods# ============================================================fig, axes = plt.subplots(2, 3, figsize=(15, 9))axes = axes.flatten()for idx, (name, (init_fn, act_fn)) in enumerate(configs.items()):    Ws = [init_fn(n_units, n_units) for _ in range(n_layers)]    stats = forward_track_variance(Ws, x, act_fn)        layers = [s[0] for s in stats]    variances = [s[2] for s in stats]        ax = axes[idx]    ax.plot(layers, variances, 'o-', linewidth=2, markersize=6)    ax.set_xlabel('Layer')    ax.set_ylabel('Variance')    ax.set_title(f'{name}\n(act={act_fn.__name__})', fontsize=11)    ax.set_yscale('log')    ax.set_ylim(1e-15, 1e10)    ax.axhline(y=1.0, color='r', linestyle='--', alpha=0.5, label='Var=1')    ax.legend(fontsize=8)    ax.grid(True, alpha=0.3)plt.suptitle('Activation Variance Through 10 Layers\n(Initialization Comparison)', fontsize=14, fontweight='bold')plt.tight_layout()plt.savefig('d:/download/6aa68b099b47c2ba093ab510/cs231n-site/notebooks/part2-cnn-vision/lecture-07-training-nn/init_variance.png', dpi=150, bbox_inches='tight')plt.show()print("Figure 1 saved: init_variance.png")

In [12]:
# ============================================================# Experiment: Weight distribution histograms# ============================================================fig, axes = plt.subplots(2, 3, figsize=(15, 9))axes = axes.flatten()n_in, n_out = 512, 256init_methods = {    'Zeros': init_zeros(n_in, n_out),    'Normal(0.01)': init_normal(n_in, n_out),    'Xavier Uniform': init_xavier_uniform(n_in, n_out),    'Xavier Normal': init_xavier_normal(n_in, n_out),    'He Normal': init_he_normal(n_in, n_out),    'He Uniform': init_he_uniform(n_in, n_out),}for idx, (name, W) in enumerate(init_methods.items()):    ax = axes[idx]    ax.hist(W.flatten(), bins=80, density=True, color='steelblue', edgecolor='white', alpha=0.8)    ax.set_title(f'{name}\nstd={np.std(W):.4f}, range=[{W.min():.4f}, {W.max():.4f}]', fontsize=10)    ax.set_xlabel('Weight Value')    ax.set_ylabel('Density')    ax.grid(True, alpha=0.3)plt.suptitle('Weight Distributions (512x256 layer)', fontsize=14, fontweight='bold')plt.tight_layout()plt.savefig('d:/download/6aa68b099b47c2ba093ab510/cs231n-site/notebooks/part2-cnn-vision/lecture-07-training-nn/init_dist.png', dpi=150, bbox_inches='tight')plt.show()print("Figure 2 saved: init_dist.png")

In [13]:
# ============================================================# Experiment: Gradient flow analysis# ============================================================def backward_track_gradient(Ws, x, activation, grad_output=None):    # Track gradient magnitude through backward pass    activations = [x.copy()]    pre_acts = []    a = x.copy()    for W in Ws:        z = a @ W        pre_acts.append(z)        a = activation(z)        activations.append(a)        if grad_output is None:        grad = np.ones_like(activations[-1])    else:        grad = grad_output        grad_norms = [np.linalg.norm(grad)]    for i in range(len(Ws) - 1, -1, -1):        if activation is relu:            grad = grad * (pre_acts[i] > 0).astype(float)        elif activation is tanh:            grad = grad * (1 - activations[i + 1] ** 2)        grad = grad @ Ws[i].T        grad_norms.append(np.linalg.norm(grad))        grad_norms.reverse()    return grad_normsfig, ax = plt.subplots(figsize=(10, 6))for name, (init_fn, act_fn) in [('Xavier+tanh', (init_xavier_normal, tanh)),                                   ('He+ReLU', (init_he_normal, relu)),                                  ('Normal(0.01)+tanh', (init_normal, tanh)),                                  ('N(0,1)+ReLU', (lambda a, b: np.random.randn(a, b), relu))]:    Ws = [init_fn(n_units, n_units) for _ in range(n_layers)]    grad_norms = backward_track_gradient(Ws, x, act_fn)    layers = list(range(len(grad_norms)))    ax.plot(layers, grad_norms, 'o-', linewidth=2, markersize=5, label=name)ax.set_xlabel('Layer (0=input)')ax.set_ylabel('Gradient Norm')ax.set_title('Gradient Flow Through 10 Layers', fontsize=14, fontweight='bold')ax.set_yscale('log')ax.legend(fontsize=10)ax.grid(True, alpha=0.3)plt.tight_layout()plt.savefig('d:/download/6aa68b099b47c2ba093ab510/cs231n-site/notebooks/part2-cnn-vision/lecture-07-training-nn/grad_flow.png', dpi=150, bbox_inches='tight')plt.show()print("Figure 3 saved: grad_flow.png")

## 5. 批归一化 (Batch Normalization)：原理与实现### 直觉批归一化通过在每个层对激活值进行标准化，解决了内部协变量偏移问题。它使得：1. 每层输入分布更稳定2. 允许使用更大的学习率3. 减少对初始化的敏感度4. 具有轻微的正则化效果### 手动计算对于mini-batch B = {x_1, ..., x_m}：**步骤1：计算批次均值** mu_B = (1/m) * sum(x_i)**步骤2：计算批次方差** sigma_B^2 = (1/m) * sum((x_i - mu_B)^2)**步骤3：标准化** x_hat_i = (x_i - mu_B) / sqrt(sigma_B^2 + eps)**步骤4：缩放和平移** y_i = gamma * x_hat_i + beta其中 gamma 和 beta 是可学习参数。**数值示例：**- 输入: x = [1.0, 2.0, 3.0, 4.0]- 均值: mu = 2.5- 方差: sigma^2 = 1.25- 标准化: x_hat = [-1.34, -0.45, 0.45, 1.34]- 若 gamma=2, beta=1: y = [-1.68, 0.10, 1.90, 3.68]

In [15]:
# ============================================================# Batch Normalization Implementation# ============================================================class BatchNorm:    # Batch Normalization for fully connected layers    def __init__(self, dim, momentum=0.9, eps=1e-5):        self.gamma = np.ones(dim)        self.beta = np.zeros(dim)        self.momentum = momentum        self.eps = eps        self.running_mean = np.zeros(dim)        self.running_var = np.ones(dim)        def forward(self, x, training=True):        if training:            batch_mean = x.mean(axis=0)            batch_var = x.var(axis=0)            self.running_mean = self.momentum * self.running_mean + (1 - self.momentum) * batch_mean            self.running_var = self.momentum * self.running_var + (1 - self.momentum) * batch_var            x_norm = (x - batch_mean) / np.sqrt(batch_var + self.eps)        else:            x_norm = (x - self.running_mean) / np.sqrt(self.running_var + self.eps)        return self.gamma * x_norm + self.beta        def backward(self, x, dout):        N = x.shape[0]        batch_mean = x.mean(axis=0)        batch_var = x.var(axis=0)        x_norm = (x - batch_mean) / np.sqrt(batch_var + self.eps)                dgamma = (dout * x_norm).sum(axis=0)        dbeta = dout.sum(axis=0)                dx_norm = dout * self.gamma        dvar = (dx_norm * (x - batch_mean) * -0.5 * (batch_var + self.eps)**(-1.5)).sum(axis=0)        dmean = (dx_norm * (-1.0 / np.sqrt(batch_var + self.eps))).sum(axis=0) + dvar * (-2.0 * (x - batch_mean)).mean(axis=0)        dx = dx_norm / np.sqrt(batch_var + self.eps) + dvar * 2.0 * (x - batch_mean) / N + dmean / N                return dx, dgamma, dbeta# Test batch normnp.random.seed(42)bn = BatchNorm(64)x_test = np.random.randn(32, 64) * 3 + 5  # mean=5, std=3out = bn.forward(x_test, training=True)print(f"Input  - mean: {x_test.mean():.4f}, std: {x_test.std():.4f}")print(f"Output - mean: {out.mean():.4f}, std: {out.std():.4f}")print("BatchNorm reduces mean to ~0 and allows gamma/beta to control std")

In [16]:
# ============================================================# Experiment: BN effect on deep network training# ============================================================def train_simple_net(use_bn=False, n_epochs=200, lr=0.01, depth=8, width=128):    # Train a deep MLP with/without BN on synthetic data    np.random.seed(42)    N = 500    X = np.random.randn(N, 2)    y = (X[:, 0] * X[:, 1] > 0).astype(int)        Ws = [init_he_normal(2, width)] + [init_he_normal(width, width) for _ in range(depth - 2)] + [init_he_normal(width, 2)]    bns = [BatchNorm(width) for _ in range(depth - 1)]        losses = []    for epoch in range(n_epochs):        a = X        cache = [a]        for i in range(depth - 1):            z = a @ Ws[i]            if use_bn:                z = bns[i].forward(z, training=True)            a = relu(z)            cache.append(a)        out = a @ Ws[-1]                probs = np.exp(out - out.max(axis=1, keepdims=True))        probs /= probs.sum(axis=1, keepdims=True)        loss = -np.log(probs[np.arange(N), y]).mean()        losses.append(loss)                dout = probs.copy()        dout[np.arange(N), y] -= 1        dout /= N                dW_last = cache[-1].T @ dout        da = dout @ Ws[-1].T        for i in range(depth - 2, -1, -1):            da = da * (cache[i + 1] > 0).astype(float)            dW = cache[i].T @ da            Ws[i] -= lr * dW            da = da @ Ws[i].T        return losseslosses_no_bn = train_simple_net(use_bn=False, n_epochs=200)losses_bn = train_simple_net(use_bn=True, n_epochs=200)fig, ax = plt.subplots(figsize=(10, 6))ax.plot(losses_no_bn, label='Without BN', alpha=0.8, linewidth=2)ax.plot(losses_bn, label='With BN', alpha=0.8, linewidth=2)ax.set_xlabel('Epoch')ax.set_ylabel('Loss')ax.set_title('Training Loss: BN vs No BN (8-layer MLP)', fontsize=14, fontweight='bold')ax.legend(fontsize=12)ax.grid(True, alpha=0.3)ax.set_ylim(0, 2)plt.tight_layout()plt.savefig('d:/download/6aa68b099b47c2ba093ab510/cs231n-site/notebooks/part2-cnn-vision/lecture-07-training-nn/bn_effect.png', dpi=150, bbox_inches='tight')plt.show()print("Figure 4 saved: bn_effect.png")

## 6. 动量法 (Momentum)：从物理到优化### 直觉想象一个球在损失曲面上滚动。标准SGD每步只看当前梯度，而动量法积累了历史梯度方向，像物理动量一样让球在平坦区域加速，在谷底减少振荡。### 手动计算**标准动量法：**- v_t = mu * v_{t-1} - eta * grad(theta_t)- theta_{t+1} = theta_t + v_t其中 mu 是动量系数（通常0.9），eta 是学习率。**数值示例：**- 初始 theta = 1.0, v_0 = 0, mu = 0.9, eta = 0.1- 梯度 g = 2*theta = 2.0- v_1 = 0.9 * 0 - 0.1 * 2.0 = -0.2- theta_1 = 1.0 + (-0.2) = 0.8- 下一步梯度 g = 2 * 0.8 = 1.6- v_2 = 0.9 * (-0.2) - 0.1 * 1.6 = -0.18 - 0.16 = -0.34- theta_2 = 0.8 + (-0.34) = 0.46注意 v_2 的绝对值比 v_1 大，动量在加速！### Nesterov动量v_t = mu * v_{t-1} - eta * grad(theta_t + mu * v_{t-1})Nesterov在梯度计算前"提前看一步"，在凸函数上有更好的理论收敛率。

In [18]:
# ============================================================# Optimizer Implementations# ============================================================class SGD:    def __init__(self, lr=0.01):        self.lr = lr    def update(self, params, grads):        for k in params:            params[k] -= self.lr * grads[k]        return paramsclass SGDMomentum:    def __init__(self, lr=0.01, momentum=0.9):        self.lr = lr        self.momentum = momentum        self.v = {}    def update(self, params, grads):        for k in params:            if k not in self.v:                self.v[k] = np.zeros_like(params[k])            self.v[k] = self.momentum * self.v[k] - self.lr * grads[k]            params[k] += self.v[k]        return paramsclass NesterovMomentum:    def __init__(self, lr=0.01, momentum=0.9):        self.lr = lr        self.momentum = momentum        self.v = {}    def update(self, params, grads_fn):        for k in params:            if k not in self.v:                self.v[k] = np.zeros_like(params[k])        look_ahead = {k: params[k] + self.momentum * self.v.get(k, 0) for k in params}        grads = grads_fn(look_ahead)        for k in params:            self.v[k] = self.momentum * self.v[k] - self.lr * grads[k]            params[k] += self.v[k]        return paramsclass Adam:    def __init__(self, lr=0.001, beta1=0.9, beta2=0.999, eps=1e-8):        self.lr = lr        self.beta1 = beta1        self.beta2 = beta2        self.eps = eps        self.m = {}        self.v = {}        self.t = 0    def update(self, params, grads):        self.t += 1        for k in params:            if k not in self.m:                self.m[k] = np.zeros_like(params[k])                self.v[k] = np.zeros_like(params[k])            self.m[k] = self.beta1 * self.m[k] + (1 - self.beta1) * grads[k]            self.v[k] = self.beta2 * self.v[k] + (1 - self.beta2) * (grads[k] ** 2)            m_hat = self.m[k] / (1 - self.beta1 ** self.t)            v_hat = self.v[k] / (1 - self.beta2 ** self.t)            params[k] -= self.lr * m_hat / (np.sqrt(v_hat) + self.eps)        return paramsprint("Optimizers defined: SGD, SGDMomentum, NesterovMomentum, Adam")

## 7. Adam优化器：自适应学习率### 直觉Adam结合了动量（一阶矩估计）和RMSProp（二阶矩估计），为每个参数自动调整学习率。梯度变化大的参数获得更小的有效学习率，反之亦然。### 手动计算**更新规则：**- m_t = beta1 * m_{t-1} + (1 - beta1) * g_t  (一阶矩)- v_t = beta2 * v_{t-1} + (1 - beta2) * g_t^2  (二阶矩)- m_hat = m_t / (1 - beta1^t)  (偏差修正)- v_hat = v_t / (1 - beta2^t)  (偏差修正)- theta_{t+1} = theta_t - eta * m_hat / (sqrt(v_hat) + eps)**数值示例：**- theta_0 = 1.0, beta1 = 0.9, beta2 = 0.999, eta = 0.1- g_1 = 2.0 (梯度)- m_1 = 0.9 * 0 + 0.1 * 2.0 = 0.2- v_1 = 0.999 * 0 + 0.001 * 4.0 = 0.004- 偏差修正: m_hat = 0.2 / (1 - 0.9) = 2.0, v_hat = 0.004 / (1 - 0.999) = 4.0- 更新: theta_1 = 1.0 - 0.1 * 2.0 / (2.0 + 1e-8) = 0.9注意偏差修正使初始步长不至过小！

In [20]:
# ============================================================# Experiment: Compare optimizers on loss landscape# ============================================================# Rosenbrock function: f(x, y) = (1-x)^2 + 100(y - x^2)^2def rosenbrock(params):    x, y = params['x'], params['y']    return (1 - x)**2 + 100 * (y - x**2)**2def rosenbrock_grad(params):    x, y = params['x'], params['y']    return {'x': -2*(1-x) - 400*x*(y-x**2), 'y': 200*(y-x**2)}# Run optimizersnp.random.seed(42)optimizers = {    'SGD (lr=0.001)': lambda: SGD(lr=0.001),    'Momentum (lr=0.001)': lambda: SGDMomentum(lr=0.001, momentum=0.9),    'Adam (lr=0.01)': lambda: Adam(lr=0.01),}trajectories = {}for name, opt_factory in optimizers.items():    params = {'x': -1.5, 'y': 2.5}    opt = opt_factory()    path = [(params['x'], params['y'])]    for _ in range(500):        grads = rosenbrock_grad(params)        params = opt.update(params, grads)        path.append((params['x'], params['y']))    trajectories[name] = np.array(path)fig, ax = plt.subplots(figsize=(10, 8))X_grid, Y_grid = np.meshgrid(np.linspace(-2, 2, 200), np.linspace(-1, 3, 200))Z = (1 - X_grid)**2 + 100 * (Y_grid - X_grid**2)**2ax.contour(X_grid, Y_grid, Z, levels=np.logspace(-1, 3, 20), cmap='jet', alpha=0.3)ax.plot(1, 1, 'r*', markersize=15, label='Minimum')colors = ['blue', 'green', 'orange']for (name, traj), color in zip(trajectories.items(), colors):    ax.plot(traj[:, 0], traj[:, 1], color=color, linewidth=2, label=name, alpha=0.8)    ax.plot(traj[0, 0], traj[0, 1], 'o', color=color, markersize=8)ax.set_xlabel('x')ax.set_ylabel('y')ax.set_title('Optimizer Comparison on Rosenbrock Function', fontsize=14, fontweight='bold')ax.legend(fontsize=10)ax.set_xlim(-2, 2)ax.set_ylim(-1, 3)plt.tight_layout()plt.savefig('d:/download/6aa68b099b47c2ba093ab510/cs231n-site/notebooks/part2-cnn-vision/lecture-07-training-nn/optimizer_compare.png', dpi=150, bbox_inches='tight')plt.show()print("Figure 5 saved: optimizer_compare.png")

In [21]:
# ============================================================# Experiment: Convergence speed comparison# ============================================================np.random.seed(42)optimizers2 = {    'SGD': lambda: SGD(lr=0.001),    'Momentum': lambda: SGDMomentum(lr=0.001, momentum=0.9),    'Adam': lambda: Adam(lr=0.001),}fig, ax = plt.subplots(figsize=(10, 6))for name, opt_factory in optimizers2.items():    params = {'x': -1.5, 'y': 2.5}    opt = opt_factory()    losses = [rosenbrock(params)]    for _ in range(300):        grads = rosenbrock_grad(params)        params = opt.update(params, grads)        losses.append(rosenbrock(params))    ax.plot(losses, linewidth=2, label=name)ax.set_xlabel('Iteration')ax.set_ylabel('Loss (log scale)')ax.set_title('Convergence Speed on Rosenbrock', fontsize=14, fontweight='bold')ax.set_yscale('log')ax.legend(fontsize=12)ax.grid(True, alpha=0.3)plt.tight_layout()plt.savefig('d:/download/6aa68b099b47c2ba093ab510/cs231n-site/notebooks/part2-cnn-vision/lecture-07-training-nn/convergence.png', dpi=150, bbox_inches='tight')plt.show()print("Figure 6 saved: convergence.png")

## 8. 学习率调度策略### 直觉训练初期需要大学习率快速接近最优点，后期需要小学习率精细调整。学习率调度自动管理这一过程。### 常见策略**1. 阶梯衰减：** 每 N 个epoch将学习率乘以 gamma**2. 指数衰减：** eta_t = eta_0 * exp(-lambda * t)**3. 余弦退火：** eta_t = eta_min/2 * (1 + cos(pi * t / T)) + eta_min/2**4. Warmup + 余弦退火：** 先线性增大再余弦减小**手动计算（阶梯衰减）：**- eta_0 = 0.1, gamma = 0.5, N = 10- epoch 0-9: eta = 0.1- epoch 10-19: eta = 0.05- epoch 20-29: eta = 0.025

In [23]:
# ============================================================# Learning Rate Schedulers# ============================================================class StepLR:    def __init__(self, lr0=0.1, step_size=10, gamma=0.5):        self.lr0 = lr0        self.step_size = step_size        self.gamma = gamma    def get_lr(self, epoch):        return self.lr0 * (self.gamma ** (epoch // self.step_size))class ExponentialLR:    def __init__(self, lr0=0.1, decay=0.01):        self.lr0 = lr0        self.decay = decay    def get_lr(self, epoch):        return self.lr0 * np.exp(-self.decay * epoch)class CosineAnnealingLR:    def __init__(self, lr0=0.1, lr_min=0.001, T=100):        self.lr0 = lr0        self.lr_min = lr_min        self.T = T    def get_lr(self, epoch):        return self.lr_min + 0.5 * (self.lr0 - self.lr_min) * (1 + np.cos(np.pi * epoch / self.T))class WarmupCosineLR:    def __init__(self, lr0=0.1, lr_min=0.001, T=100, warmup=10):        self.lr0 = lr0        self.lr_min = lr_min        self.T = T        self.warmup = warmup    def get_lr(self, epoch):        if epoch < self.warmup:            return self.lr0 * (epoch + 1) / self.warmup        return self.lr_min + 0.5 * (self.lr0 - self.lr_min) * (1 + np.cos(np.pi * (epoch - self.warmup) / (self.T - self.warmup)))schedulers = {    'Step (gamma=0.5, step=10)': StepLR(lr0=0.1, step_size=10, gamma=0.5),    'Exponential (decay=0.02)': ExponentialLR(lr0=0.1, decay=0.02),    'Cosine Annealing': CosineAnnealingLR(lr0=0.1, lr_min=0.001, T=100),    'Warmup + Cosine': WarmupCosineLR(lr0=0.1, lr_min=0.001, T=100, warmup=10),}fig, ax = plt.subplots(figsize=(10, 6))epochs = np.arange(100)for name, sched in schedulers.items():    lrs = [sched.get_lr(e) for e in epochs]    ax.plot(epochs, lrs, linewidth=2, label=name)ax.set_xlabel('Epoch')ax.set_ylabel('Learning Rate')ax.set_title('Learning Rate Schedules', fontsize=14, fontweight='bold')ax.legend(fontsize=10)ax.grid(True, alpha=0.3)plt.tight_layout()plt.savefig('d:/download/6aa68b099b47c2ba093ab510/cs231n-site/notebooks/part2-cnn-vision/lecture-07-training-nn/lr_schedule.png', dpi=150, bbox_inches='tight')plt.show()print("Figure 7 saved: lr_schedule.png")

## 9. 梯度检验 (Gradient Checking)### 直觉梯度检验通过数值方法验证解析梯度的正确性。核心思想是使用有限差分近似梯度，然后与反向传播计算的梯度比较。### 手动计算**数值梯度（中心差分）：**grad ~= (f(x + h) - f(x - h)) / (2h)**相对误差：**rel_error = ||g_analytic - g_numerical|| / (||g_analytic|| + ||g_numerical||)**判定标准：**- 相对误差 < 1e-7：梯度正确- 相对误差在 1e-4 到 1e-7：可能正确，需要检查- 相对误差 > 1e-4：梯度可能有误**数值示例：**- f(x) = x^2, x = 3.0, h = 0.001- 解析梯度: f'(3) = 6.0- 数值梯度: (3.001^2 - 2.999^2) / 0.002 = (9.006001 - 8.994001) / 0.002 = 6.0- 相对误差: 0 （完美！）

In [25]:
# ============================================================# Gradient Checking Implementation# ============================================================def numerical_gradient(f, x, h=1e-5):    # Compute numerical gradient using central difference    grad = np.zeros_like(x)    it = np.nditer(x, flags=['multi_index'], op_flags=['readwrite'])    while not it.finished:        idx = it.multi_index        old_val = x[idx]        x[idx] = old_val + h        fx_plus = f(x)        x[idx] = old_val - h        fx_minus = f(x)        x[idx] = old_val        grad[idx] = (fx_plus - fx_minus) / (2 * h)        it.iternext()    return graddef rel_error(x, y):    # Compute relative error between two arrays    return np.max(np.abs(x - y) / (np.maximum(1e-8, np.abs(x) + np.abs(y))))# Test: quadratic function f(x) = sum(x^2)np.random.seed(42)x = np.random.randn(5, 5)f = lambda x: np.sum(x ** 2)analytic_grad = 2 * xnumerical_grad = numerical_gradient(f, x.copy())print("=== Gradient Check: f(x) = sum(x^2) ===")print(f"Analytic grad sample: {analytic_grad[0, :3]}")print(f"Numerical grad sample: {numerical_grad[0, :3]}")print(f"Relative error: {rel_error(analytic_grad, numerical_grad):.2e}")print(f"Pass threshold (< 1e-7): {rel_error(analytic_grad, numerical_grad) < 1e-7}")

In [26]:
# ============================================================# Gradient check on softmax loss# ============================================================np.random.seed(42)N, D, C = 20, 10, 3  # 20 samples, 10 features, 3 classesX = np.random.randn(N, D)W = np.random.randn(D, C)y = np.random.randint(0, C, N)# Analytic gradientscores = X @ Wshifted = scores - np.max(scores, axis=1, keepdims=True)exp_scores = np.exp(shifted)probs = exp_scores / exp_scores.sum(axis=1, keepdims=True)loss_val = -np.sum(np.log(probs[np.arange(N), y] + 1e-12)) / Ndscores = probs.copy()dscores[np.arange(N), y] -= 1dscores /= NdW_analytic = X.T @ dscores# Numerical gradientdef softmax_loss_w(W_param):    s = X @ W_param    s = s - s.max(1, keepdims=True)    p = np.exp(s) / np.exp(s).sum(1, keepdims=True)    return -np.sum(np.log(p[np.arange(N), y] + 1e-12)) / NdW_numerical = numerical_gradient(softmax_loss_w, W.copy())print("=== Gradient Check: Softmax Loss ===")print(f"Loss value: {loss_val:.6f}")print(f"Relative error (W): {rel_error(dW_analytic, dW_numerical):.2e}")print(f"Pass threshold (< 1e-7): {rel_error(dW_analytic, dW_numerical) < 1e-7}")

In [27]:
# ============================================================# Gradient check: BatchNorm backward# ============================================================np.random.seed(42)N_bn, D_bn = 8, 5x_bn = np.random.randn(N_bn, D_bn)gamma_bn = np.random.randn(D_bn)beta_bn = np.random.randn(D_bn)dout_bn = np.random.randn(N_bn, D_bn)bn_test = BatchNorm(D_bn)# Analytic backwardout_bn = bn_test.forward(x_bn, training=True)dx_analytic, dgamma_analytic, dbeta_analytic = bn_test.backward(x_bn, dout_bn)# Numerical gradientsdef bn_loss_x(x_param):    bn_temp = BatchNorm(D_bn)    bn_temp.gamma = gamma_bn    bn_temp.beta = beta_bn    return np.sum(bn_temp.forward(x_param, training=True) * dout_bn)dx_num = numerical_gradient(bn_loss_x, x_bn.copy())def bn_loss_gamma(g_param):    bn_temp = BatchNorm(D_bn)    bn_temp.gamma = g_param    bn_temp.beta = beta_bn    return np.sum(bn_temp.forward(x_bn, training=True) * dout_bn)dgamma_num = numerical_gradient(bn_loss_gamma, gamma_bn.copy())print("=== Gradient Check: BatchNorm ===")print(f"dx relative error:     {rel_error(dx_analytic, dx_num):.2e}")print(f"dgamma relative error: {rel_error(dgamma_analytic, dgamma_num):.2e}")print(f"All < 1e-6: {rel_error(dx_analytic, dx_num) < 1e-6 and rel_error(dgamma_analytic, dgamma_num) < 1e-6}")

## 10. 训练合理性检查 (Sanity Checks)### 训练前检查清单1. **过拟合小批量数据**：在少量样本上应该能达到接近零的损失2. **初始损失检查**：初始损失应接近随机猜测的损失 (-log(1/C))3. **初始化检查**：激活值不应全部为零或爆炸4. **梯度范围检查**：各层梯度量级不应相差太大### 手动验证对于C=10的分类问题：- 随机猜测的期望损失 = -log(1/10) = ln(10) = 2.303- 如果初始损失远大于2.303，可能初始化有问题- 如果训练10个样本1000次迭代后损失不下降，可能有bug

In [29]:
# ============================================================# Sanity Check 1: Initial loss should be ~ln(C)# ============================================================np.random.seed(42)N, D, C = 50, 20, 10X_sc = np.random.randn(N, D)W_sc = init_he_normal(D, C) * 0.01  # Small inity_sc = np.random.randint(0, C, N)scores = X_sc @ W_scshifted = scores - scores.max(1, keepdims=True)probs = np.exp(shifted) / np.exp(shifted).sum(1, keepdims=True)initial_loss = -np.sum(np.log(probs[np.arange(N), y_sc] + 1e-12)) / Nprint("=== Sanity Check 1: Initial Loss ===")print(f"Number of classes: {C}")print(f"Expected (random) loss: -ln(1/{C}) = {-np.log(1.0/C):.4f}")print(f"Actual initial loss:   {initial_loss:.4f}")print(f"Ratio (should be ~1.0): {initial_loss / (-np.log(1.0/C)):.4f}")

In [30]:
# ============================================================# Sanity Check 2: Overfit small batch# ============================================================np.random.seed(42)N_small, D_s, C_s = 10, 20, 3X_small = np.random.randn(N_small, D_s)y_small = np.random.randint(0, C_s, N_small)W1_s = init_he_normal(D_s, 64)W2_s = init_he_normal(64, C_s)losses_sanity = []for epoch in range(1000):    h = relu(X_small @ W1_s)    scores = h @ W2_s    shifted = scores - scores.max(1, keepdims=True)    probs = np.exp(shifted) / np.exp(shifted).sum(1, keepdims=True)    loss = -np.sum(np.log(probs[np.arange(N_small), y_small] + 1e-12)) / N_small    losses_sanity.append(loss)        dscores = probs.copy()    dscores[np.arange(N_small), y_small] -= 1    dscores /= N_small    dW2 = h.T @ dscores    dh = dscores @ W2_s.T    dW1 = X_small.T @ (dh * (h > 0))        W1_s -= 0.1 * dW1    W2_s -= 0.1 * dW2print("=== Sanity Check 2: Overfit Small Batch ===")print(f"Initial loss: {losses_sanity[0]:.4f}")print(f"Final loss:   {losses_sanity[-1]:.4f}")print(f"Loss reduction: {losses_sanity[0] / max(losses_sanity[-1], 1e-10):.1f}x")print(f"Should be near 0 if implementation is correct: {'PASS' if losses_sanity[-1] < 0.01 else 'FAIL'}")fig, ax = plt.subplots(figsize=(8, 5))ax.plot(losses_sanity, linewidth=2)ax.set_xlabel('Iteration')ax.set_ylabel('Loss')ax.set_title('Sanity Check: Overfit 10 Samples', fontsize=14, fontweight='bold')ax.set_yscale('log')ax.grid(True, alpha=0.3)plt.tight_layout()plt.savefig('d:/download/6aa68b099b47c2ba093ab510/cs231n-site/notebooks/part2-cnn-vision/lecture-07-training-nn/sanity_check.png', dpi=150, bbox_inches='tight')plt.show()print("Figure 8 saved: sanity_check.png")

## 11. 综合实验：完整训练流程将上述所有技术整合，在一个完整训练流程中对比不同配置的效果。

In [32]:
# ============================================================# Comprehensive Experiment: Full training pipeline# ============================================================np.random.seed(42)# Generate synthetic dataset (2-class spiral)N = 300r = np.linspace(0, 5, N // 2)theta1 = np.linspace(0, 3*np.pi, N // 2) + np.random.randn(N//2) * 0.3theta2 = np.linspace(0, 3*np.pi, N // 2) + np.pi + np.random.randn(N//2) * 0.3X1 = np.stack([r * np.cos(theta1), r * np.sin(theta1)], axis=1)X2 = np.stack([r * np.cos(theta2), r * np.sin(theta2)], axis=1)X_data = np.vstack([X1, X2])y_data = np.array([0]*len(X1) + [1]*len(X2))idx = np.random.permutation(N)X_data, y_data = X_data[idx], y_data[idx]D_net, H_net, C_net = 2, 64, 2configs_full = {    'SGD + He': {'opt': SGD(lr=0.05), 'init': 'he'},    'Momentum + He': {'opt': SGDMomentum(lr=0.05, momentum=0.9), 'init': 'he'},    'Adam + He': {'opt': Adam(lr=0.01), 'init': 'he'},    'Adam + Xavier': {'opt': Adam(lr=0.01), 'init': 'xavier'},}results = {}for name, cfg in configs_full.items():    np.random.seed(42)    init_fn = init_he_normal if cfg['init'] == 'he' else init_xavier_normal    W1, W2 = init_fn(D_net, H_net), init_fn(H_net, C_net)    b1, b2 = np.zeros(H_net), np.zeros(C_net)    opt = cfg['opt']    params = {'W1': W1, 'W2': W2, 'b1': b1, 'b2': b2}    losses_list = []    accs_list = []        for epoch in range(300):        h = relu(X_data @ params['W1'] + params['b1'])        scores = h @ params['W2'] + params['b2']        shifted = scores - scores.max(1, keepdims=True)        probs = np.exp(shifted) / np.exp(shifted).sum(1, keepdims=True)        loss = -np.sum(np.log(probs[np.arange(N), y_data] + 1e-12)) / N        losses_list.append(loss)                pred = scores.argmax(1)        acc = (pred == y_data).mean()        accs_list.append(acc)                dscores = probs.copy()        dscores[np.arange(N), y_data] -= 1        dscores /= N        grads = {            'W2': h.T @ dscores,            'b2': dscores.sum(0),            'W1': X_data.T @ (dscores @ params['W2'].T * (h > 0)),            'b1': (dscores @ params['W2'].T * (h > 0)).sum(0),        }        params = opt.update(params, grads)        results[name] = {'loss': losses_list, 'acc': accs_list}fig, axes = plt.subplots(1, 2, figsize=(14, 5))for name, res in results.items():    axes[0].plot(res['loss'], linewidth=2, label=name)    axes[1].plot(res['acc'], linewidth=2, label=name)axes[0].set_xlabel('Epoch')axes[0].set_ylabel('Loss')axes[0].set_title('Training Loss', fontweight='bold')axes[0].legend()axes[0].grid(True, alpha=0.3)axes[1].set_xlabel('Epoch')axes[1].set_ylabel('Accuracy')axes[1].set_title('Training Accuracy', fontweight='bold')axes[1].legend()axes[1].grid(True, alpha=0.3)plt.suptitle('Full Training Pipeline: Optimizer + Init Comparison', fontsize=14, fontweight='bold')plt.tight_layout()plt.savefig('d:/download/6aa68b099b47c2ba093ab510/cs231n-site/notebooks/part2-cnn-vision/lecture-07-training-nn/full_training.png', dpi=150, bbox_inches='tight')plt.show()print("Figure 9 saved: full_training.png")for name, res in results.items():    print(f"{name}: final_loss={res['loss'][-1]:.4f}, final_acc={res['acc'][-1]:.4f}")

## 11.5 正则化与权重衰减### 直觉权重衰减（L2正则化）通过在损失函数中添加权重平方和的惩罚项，防止过拟合：L_total = L_data + lambda * sum(W^2)AdamW将权重衰减与梯度更新解耦，比标准Adam+L2效果更好：- 标准Adam+L2：梯度中包含正则化项，受自适应学习率影响- AdamW：权重衰减独立于梯度更新，直接乘到参数上### 手动计算权重为 W = [0.5, -0.3, 0.8]，权重衰减系数 wd = 0.01：- L2正则化项：0.01 * (0.25 + 0.09 + 0.64) = 0.0098- AdamW更新：W -= lr * (adam_update + wd * W)- 效果：W 会被持续拉向零，但不是通过梯度

In [34]:
# ============================================================# Demonstrate weight decay effect# ============================================================class AdamW:    def __init__(self, lr=0.001, beta1=0.9, beta2=0.999, eps=1e-8, weight_decay=0.01):        self.lr = lr        self.beta1 = beta1        self.beta2 = beta2        self.eps = eps        self.weight_decay = weight_decay        self.m = {}        self.v = {}        self.t = 0    def update(self, params, grads):        self.t += 1        for k in params:            if k not in self.m:                self.m[k] = np.zeros_like(params[k])                self.v[k] = np.zeros_like(params[k])            self.m[k] = self.beta1 * self.m[k] + (1 - self.beta1) * grads[k]            self.v[k] = self.beta2 * self.v[k] + (1 - self.beta2) * (grads[k] ** 2)            m_hat = self.m[k] / (1 - self.beta1 ** self.t)            v_hat = self.v[k] / (1 - self.beta2 ** self.t)            # Decoupled weight decay            params[k] -= self.lr * (m_hat / (np.sqrt(v_hat) + self.eps) + self.weight_decay * params[k])        return params# Compare Adam vs AdamW weight norms over trainingnp.random.seed(42)W_adam = np.random.randn(50, 10) * 0.5W_adamw = W_adam.copy()adam = Adam(lr=0.01)adamw = AdamW(lr=0.01, weight_decay=0.05)norms_adam = [np.linalg.norm(W_adam)]norms_adamw = [np.linalg.norm(W_adamw)]for _ in range(200):    # Simulate gradient (random + signal)    grad = np.random.randn(50, 10) * 0.1 + W_adam * 0.01    params_a = {'W': W_adam}    params_w = {'W': W_adamw}    params_a = adam.update(params_a, {'W': grad})    params_w = adamw.update(params_w, {'W': grad})    W_adam = params_a['W']    W_adamw = params_w['W']    norms_adam.append(np.linalg.norm(W_adam))    norms_adamw.append(np.linalg.norm(W_adamw))fig, ax = plt.subplots(figsize=(10, 5))ax.plot(norms_adam, label='Adam (no decay)', linewidth=2)ax.plot(norms_adamw, label='AdamW (wd=0.05)', linewidth=2)ax.set_xlabel('Iteration')ax.set_ylabel('Weight Norm')ax.set_title('Weight Norm: Adam vs AdamW', fontsize=14, fontweight='bold')ax.legend(fontsize=12)ax.grid(True, alpha=0.3)plt.tight_layout()plt.savefig('d:/download/6aa68b099b47c2ba093ab510/cs231n-site/notebooks/part2-cnn-vision/lecture-07-training-nn/adamw.png', dpi=150, bbox_inches='tight')plt.show()print("Figure 10 saved: adamw.png")print(f"Final weight norm - Adam: {norms_adam[-1]:.4f}")print(f"Final weight norm - AdamW: {norms_adamw[-1]:.4f}")

## 12. 关键要点总结| 技术 | 核心思想 | 适用场景 ||---|---|---|| Xavier初始化 | 保持方差=1/n | tanh/sigmoid || He初始化 | 方差=2/n补偿ReLU | ReLU网络 || 批归一化 | 标准化每层输入 | 深层网络 || 动量法 | 积累历史梯度 | 优化曲面有噪声 || Adam | 自适应每参数学习率 | 通用首选 || 梯度检验 | 数值验证解析梯度 | 调试阶段 || 学习率调度 | 训练后期降低学习率 | 精细调优 || 合理性检查 | 过拟合小批量 | 训练前验证 |**经验法则：**1. 总是先用Adam(lr=0.001)快速原型2. 深层网络使用He初始化 + ReLU3. 添加BatchNorm可以大幅提升训练稳定性4. 使用余弦退火或阶梯衰减5. 训练前务必执行合理性检查

## 13. 作业### 作业1：实现RMSProp优化器RMSProp是对Adagrad的改进，使用指数加权移动平均代替累积平方梯度：- v_t = rho * v_{t-1} + (1 - rho) * g_t^2- theta_{t+1} = theta_t - eta / (sqrt(v_t) + eps) * g_t**要求：**1. 实现RMSProp类2. 在Rosenbrock函数上与SGD、Momentum、Adam对比3. 测试不同的 rho 值（0.9, 0.95, 0.99）4. 绘制收敛曲线

### 作业2：实现Layer Normalization与BatchNorm不同，LayerNorm在特征维度上归一化，而非批次维度：- mu_i = (1/D) * sum(x_id)- sigma_i^2 = (1/D) * sum((x_id - mu_i)^2)**要求：**1. 实现LayerNorm类（forward + backward）2. 对比BatchNorm和LayerNorm在batch_size=1和batch_size=32时的表现3. 分析为什么RNN/Transformer更倾向于使用LayerNorm4. 用梯度检验验证你的backward实现

### 作业3：学习率查找 (Learning Rate Finder)实现Leslie Smith的学习率查找方法：**算法：**1. 从极小学习率（如1e-8）开始2. 每个batch后按指数增大学习率3. 记录每个学习率对应的loss4. 当loss开始发散时停止5. 选择loss下降最快的区域对应的学习率**要求：**1. 在上面的spiral数据集上实现2. 绘制loss vs learning rate曲线3. 标注最佳学习率范围4. 比较使用找到的学习率与手动调参的效果

## 14. 参考文献1. [[Glorot and Bengio, 2010]](http://proceedings.mlr.press/v9/glorot10a.html) - Understanding the difficulty of training deep feedforward neural networks (Xavier initialization)2. [[He et al., 2015]](https://arxiv.org/abs/1502.01852) - Delving deep into rectifiers: Surpassing human-level performance on ImageNet classification (He initialization)3. [[Ioffe and Szegedy, 2015]](https://arxiv.org/abs/1502.03167) - Batch Normalization: Accelerating Deep Network Training by Reducing Internal Covariate Shift4. [[Kingma and Ba, 2015]](https://arxiv.org/abs/1412.6980) - Adam: A Method for Stochastic Optimization5. [[Sutskever et al., 2013]](http://proceedings.mlr.press/v28/sutskever13.html) - On the importance of initialization and momentum in deep learning6. [[Loshchilov and Hutter, 2019]](https://arxiv.org/abs/1711.05101) - Decoupled Weight Decay Regularization (AdamW)7. [[Smith, 2017]](https://arxiv.org/abs/1704.04289) - Cyclical Learning Rates for Training Neural Networks8. [[Loshchilov and Hutter, 2017]](https://arxiv.org/abs/1608.03983) - SGDR: Stochastic Gradient Descent with Warm Restarts9. [[Ba et al., 2016]](https://arxiv.org/abs/1607.06450) - Layer Normalization10. [[CS231n, Stanford]](https://cs231n.github.io/) - CS231n: Deep Learning for Computer Vision

---

## 参考代码实现

以下 GitHub 仓库提供了本节内容的完整代码实现，建议结合学习：

- **[labmlai/annotated_deep_learning_paper_implementations](https://github.com/labmlai/annotated_deep_learning_paper_implementations)** (67433 stars): 权重初始化、Adam、学习率调度带注释实现
  - 仓库地址: https://github.com/labmlai/annotated_deep_learning_paper_implementations

- **[seloufian/Deep-Learning-Computer-Vision](https://github.com/seloufian/Deep-Learning-Computer-Vision)** (135 stars): CS231n+EECS498 训练技巧综合解答
  - 仓库地址: https://github.com/seloufian/Deep-Learning-Computer-Vision


> 标注说明: 以上仓库按热度排序，优先推荐 stars 最多的实现。

